In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown
from funciones import obtener_fecha_hoy

In [2]:
import sys, pathlib
# Sube hasta la raíz del repo (donde vive init_agents.py) y la agrega al path
_root = pathlib.Path.cwd()
while not (_root / "init_agents.py").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.append(str(_root))

import init_agents  # ejecuta set_tracing_export_api_key(OPENAI_TRACING_API_KEY)

In [3]:
load_dotenv(override=True)

True

In [4]:
obtener_fecha_hoy()

datetime.date(2026, 8, 7)

In [5]:
# Ahora usemos nuestro servidor de cuentas como servidor MCP

params = {"command": "uv", "args": ["run", "jp_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [6]:
mcp_tools

[Tool(name='get_fecha', title=None, description='Obtiene la fecha de hoy.', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_fechaArguments', 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='sumar_numeros', title=None, description='Suma dos números.\n    Args:\n        a: El primer número a sumar\n        b: El segundo número a sumar', inputSchema={'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'sumar_numerosArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'sumar_numerosOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='resolver_ecuacion_cuadratica', title=None, description='Resuelve ax² + bx + c = 0\n    Args:\n        a, b, c: Coeficientes de la ecu

In [8]:
instructions = "Eres un asistente util."
request = "cuánto es 12.5+13.5?"
model = "gpt-4.1-mini"

In [18]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="ejercicioMCP", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("ejercicioMCP"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))
    


12.5 + 13.5 es igual a 26.0. ¿Hay algo más en lo que pueda ayudarte?

In [19]:
import time

params = {"command": "uv", "args": ["run", "jp_server.py"]}

# Mide setup del servidor
start = time.time()
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    setup_time = time.time() - start
    print(f"MCP setup: {setup_time:.2f}s")
    
    # Mide ejecución del agent
    start = time.time()
    agent = Agent(name="ejercicioMCP", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("ejercicioMCP"):
        result = await Runner.run(agent, request)
    exec_time = time.time() - start
    print(f"Agent execution: {exec_time:.2f}s")
    
    display(Markdown(result.final_output))

MCP setup: 0.37s
Agent execution: 4.62s


12.5 + 13.5 es igual a 26.0. ¿Quieres que te ayude con algo más?

In [16]:
import time

request = """Ajusta un polinomio de grado 3 a estos puntos:
(1, 2.1), (2, 4.1), (3, 8.9), (4, 15.8), (5, 25.2)
"""

# Sin MCP - pregunta directa
start = time.time()
result_direct = await Runner.run(
    Agent(name="direct", instructions=instructions, model=model), 
    request
)
direct_time = time.time() - start
print(f"Direct: {direct_time:.2f}s")

# Con MCP
start = time.time()
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="ejercicioMCP", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("ejercicioMCP"):
        result_mcp = await Runner.run(agent, request)
mcp_time = time.time() - start
print(f"MCP: {mcp_time:.2f}s")
print(f"Overhead: {mcp_time - direct_time:.2f}s")

print("\n=== DIRECT ===")
print(result_direct.final_output)
print("\n=== MCP ===")
print(result_mcp.final_output)

Direct: 50.98s
MCP: 8.45s
Overhead: -42.53s

=== DIRECT ===
Para ajustar un polinomio de grado 3 (cúbico) a los puntos dados \((x_i, y_i)\):
\[
(1, 2.1), (2, 4.1), (3, 8.9), (4, 15.8), (5, 25.2),
\]
buscamos un polinomio de la forma:
\[
P(x) = a_3 x^3 + a_2 x^2 + a_1 x + a_0.
\]

El ajuste se puede hacer usando mínimos cuadrados.

---

### Paso 1: Plantear el sistema

Para cada punto \((x_i, y_i)\):
\[
a_3 x_i^3 + a_2 x_i^2 + a_1 x_i + a_0 = y_i.
\]

Esto genera un sistema lineal (en forma matrix):
\[
\begin{bmatrix}
1^3 & 1^2 & 1 & 1 \\
2^3 & 2^2 & 2 & 1 \\
3^3 & 3^2 & 3 & 1 \\
4^3 & 4^2 & 4 & 1 \\
5^3 & 5^2 & 5 & 1 \\
\end{bmatrix}
\begin{bmatrix}
a_3 \\ a_2 \\ a_1 \\ a_0
\end{bmatrix}
=
\begin{bmatrix}
2.1 \\ 4.1 \\ 8.9 \\ 15.8 \\ 25.2
\end{bmatrix}
\]

---

### Paso 2: Definir matrices (X y Y)
\[
X = \begin{bmatrix}
1 & 1 & 1 & 1 \\
8 & 4 & 2 & 1 \\
27 & 9 & 3 & 1 \\
64 & 16 & 4 & 1 \\
125 & 25 & 5 & 1
\end{bmatrix},
\quad
Y = \begin{bmatrix}
2.1 \\ 4.1 \\ 8.9 \\ 15.8 \\ 25.2
\end{

In [15]:
print(result_mcp.final_output, "\n\n", result_direct.final_output)


El polinomio de grado 3 ajustado a los puntos dados es:

y = -0.0250x^3 + 1.4321x^2 - 2.0429x + 2.7200

El coeficiente de determinación R² es aproximadamente 0.99995, lo que indica un ajuste muy bueno. ¿Quieres que te ayude con algo más? 

 12.5 + 13.5 es 26.
